# Centaur 8B 불러오기

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.55.4
!pip install --no-deps trl==0.22.2

In [ ]:
from unsloth import FastLanguageModel
import transformers

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
### LLM ###
model, tokenizer = FastLanguageModel.from_pretrained(
  model_name = "marcelbinz/Llama-3.1-Minitaur-8B-adapter",
  max_seq_length = 32768,
  dtype = None,
  load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

pipe = transformers.pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            trust_remote_code=True,
            pad_token_id=0,
            do_sample=True,
            temperature=1.0,
            max_new_tokens=1,
)

==((====))==  Unsloth 2025.11.2: Fast Llama patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/83.9M [00:00<?, ?B/s]

Unsloth 2025.11.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.
Device set to use cuda:0


# Test Example

In [ ]:
prompt = "You will be presented with triplets of objects, which will be assigned to the keys H, Y, and E.\n" \
  "In each trial, please indicate which object you think is the odd one out by pressing the corresponding key.\n" \
  "In other words, please choose the object that is the least similar to the other two.\n\n" \
  "H: plant, Y: chainsaw, and E: periscope. You press <<H>>.\n" \
  "H: tostada, Y: leaf, and E: sail. You press <<H>>.\n" \
  "H: clock, Y: crystal, and E: grate. You press <<Y>>.\n" \
  "H: barbed wire, Y: kale, and E: sweater. You press <<E>>.\n" \
  "H: raccoon, Y: toothbrush, and E: ice. You press <<"

print(prompt)

You will be presented with triplets of objects, which will be assigned to the keys H, Y, and E.
In each trial, please indicate which object you think is the odd one out by pressing the corresponding key.
In other words, please choose the object that is the least similar to the other two.

H: plant, Y: chainsaw, and E: periscope. You press <<H>>.
H: tostada, Y: leaf, and E: sail. You press <<H>>.
H: clock, Y: crystal, and E: grate. You press <<Y>>.
H: barbed wire, Y: kale, and E: sweater. You press <<E>>.
H: raccoon, Y: toothbrush, and E: ice. You press <<


In [ ]:
choice = pipe(prompt)[0]['generated_text'][len(prompt):]
print(choice)

H


# Normative Theory (종이 50번 접기)

In [ ]:
prompt = """You are taking part in a reasoning experiment.
  In each trial, you will be presented with a multiple-choice question.
  Please choose the answer that seems most reasonable to you by pressing the corresponding key.
  Only output your choice as <<1>>, <<2>>, <<3>>, <<4>>, or <<5>> with no explanation.
  Question:
  There is a very large sheet of paper.
  You fold it in half, then fold it in half again.
  If you repeat this process 50 times, how thick will the paper be?
  1: The height of a desk
  2: The height of a ceiling
  3: The height of the 63 Building
  4: The height of Mount Everest
  5: The distance from the Earth to the Sun
  You press <<"""

print(prompt)

You are taking part in a reasoning experiment. 
  In each trial, you will be presented with a multiple-choice question. 
  Please choose the answer that seems most reasonable to you by pressing the corresponding key. 
  Only output your choice as <<1>>, <<2>>, <<3>>, <<4>>, or <<5>> with no explanation. 
  Question: 
  There is a very large sheet of paper. 
  You fold it in half, then fold it in half again. 
  If you repeat this process 50 times, how thick will the paper be? 
  1: The height of a desk 
  2: The height of a ceiling 
  3: The height of the 63 Building 
  4: The height of Mount Everest 
  5: The distance from the Earth to the Sun 
  You press <<


In [ ]:
choice = pipe(prompt)[0]['generated_text'][len(prompt):]
print(choice)

2


# RH 1

In [ ]:
prompt = """You are taking part in a reasoning experiment.
In each trial, you will read a short description of a person and choose the statement that seems least likely to describe that person.
Please choose as a human would — based on intuition, not strict logic.
Output only your choice in the format <<1>>, <<2>>, or <<3>> with no explanation.
Description:
Mr. Kim Young-min is a 31-year-old single man with a very outgoing personality.
He majored in philosophy while attending university and was actively involved in various student activities.
He is very interested in social issues and goes to rural areas for volunteer work every summer vacation.
Question:
Which of the following statements is least likely to describe Mr. Kim Young-min?
1: Mr. Kim Young-min works for a loan shark company.
2: Mr. Kim Young-min supports environmental activism.
3: Mr. Kim Young-min works for a loan shark company and supports environmental activism.
You press <<"""

print(prompt)

You are taking part in a reasoning experiment.
In each trial, you will read a short description of a person and choose the statement that seems least likely to describe that person.
Please choose as a human would — based on intuition, not strict logic.
Output only your choice in the format <<1>>, <<2>>, or <<3>> with no explanation.
Description:
Mr. Kim Young-min is a 31-year-old single man with a very outgoing personality.
He majored in philosophy while attending university and was actively involved in various student activities.
He is very interested in social issues and goes to rural areas for volunteer work every summer vacation.
Question:
Which of the following statements is least likely to describe Mr. Kim Young-min?
1: Mr. Kim Young-min works for a loan shark company.
2: Mr. Kim Young-min supports environmental activism.
3: Mr. Kim Young-min works for a loan shark company and supports environmental activism.
You press <<


In [ ]:
choice = pipe(prompt)[0]['generated_text'][len(prompt):]
print(choice)

1


# RH 2

In [ ]:
prompt = """You are taking part in a reasoning experiment.
Select the answer that seems most plausible to you.
Output only <<1>>, <<2>>, or <<3>>.
A certain town is served by two hospitals.
In the larger hospital about 45 babies are born each day, and in the smaller hospital about 15 babies are born each day.
About 50 percent of all babies are boys, though the percentage varies from day to day.
Over a year, each hospital recorded the number of days when more than 60 percent of the babies born were boys.
Which hospital recorded more such days?
Question:
Which of the following statements is least likely to describe Mr. Kim Young-min?
1: The larger hospital
2: The smaller hospital
3: About the same (within 5 percent)
You press <<"""

print(prompt)

You are taking part in a reasoning experiment.
Select the answer that seems most plausible to you.
Output only <<1>>, <<2>>, or <<3>>.
A certain town is served by two hospitals.
In the larger hospital about 45 babies are born each day, and in the smaller hospital about 15 babies are born each day.
About 50 percent of all babies are boys, though the percentage varies from day to day.
Over a year, each hospital recorded the number of days when more than 60 percent of the babies born were boys.
Which hospital recorded more such days?
Question:
Which of the following statements is least likely to describe Mr. Kim Young-min?
1: The larger hospital
2: The smaller hospital
3: About the same (within 5 percent)
You press <<


In [ ]:
choice = pipe(prompt)[0]['generated_text'][len(prompt):]
print(choice)

1


# RH 3

In [ ]:
prompt = """You are taking part in a reasoning experiment.
Select who should be more confident that the urn truly contains ⅔ red balls and ⅓ white balls.
Output only <<1>> or <<2>>.
- Person A drew 5 balls: 4 red, 1 white.
- Person B drew 20 balls: 12 red, 8 white.
Question:
Who should be more confident?
1: Person A
2: Person B
You press <<"""

print(prompt)

You are taking part in a reasoning experiment.
Select who should be more confident that the urn truly contains ⅔ red balls and ⅓ white balls.
Output only <<1>> or <<2>>.
- Person A drew 5 balls: 4 red, 1 white.
- Person B drew 20 balls: 12 red, 8 white.
Question:
Who should be more confident?
1: Person A
2: Person B
You press <<


In [ ]:
choice = pipe(prompt)[0]['generated_text'][len(prompt):]
print(choice)

1


# RH 4

In [ ]:
prompt = """You are taking part in a reasoning experiment.
In this task, you will answer questions about coin flips.
For each question, choose the option that best matches your judgment.
Output only your choice as <<1>>, <<2>>, or <<3>> with no explanation.
You flip a fair coin six times and obtain the following sequence:
H – T – H – T – T – H
Question:
Do you think this sequence is more likely, less likely, or just as likely to occur as a sequence like H-H-H-T-T-T or H-H-H-H-T-H?
1: This sequence seems more likely because it has both heads and tails.
2: This sequence seems less likely because it has too many changes between heads and tails.
3: This sequence is just as likely as any other sequence of six coin flips.
You press <<"""

print(prompt)

You are taking part in a reasoning experiment.
In this task, you will answer questions about coin flips.
For each question, choose the option that best matches your judgment.
Output only your choice as <<1>>, <<2>>, or <<3>> with no explanation.
You flip a fair coin six times and obtain the following sequence:
H – T – H – T – T – H
Question:
Do you think this sequence is more likely, less likely, or just as likely to occur as a sequence like H-H-H-T-T-T or H-H-H-H-T-H?
1: This sequence seems more likely because it has both heads and tails.
2: This sequence seems less likely because it has too many changes between heads and tails.
3: This sequence is just as likely as any other sequence of six coin flips.
You press <<


In [ ]:
choice = pipe(prompt)[0]['generated_text'][len(prompt):]
print(choice)

1


# RH 5

In [ ]:
prompt = """You are taking part in a reasoning experiment.
Select the answer that seems most plausible to you.
Output only <<1>>, <<2>>, <<3>>, or <<4>>.
You roll a die ten times and get only even numbers.
What do you expect on the next roll?
1: More likely an odd number to balance out.
2: Completely independent; every number equally likely.
3: The die is probably defective.
4: You’ve used up your luck; an odd number is guaranteed next.
You press <<"""

print(prompt)

You are taking part in a reasoning experiment.
Select the answer that seems most plausible to you.
Output only <<1>>, <<2>>, <<3>>, or <<4>>.
You roll a die ten times and get only even numbers.
What do you expect on the next roll?
1: More likely an odd number to balance out.
2: Completely independent; every number equally likely.
3: The die is probably defective.
4: You’ve used up your luck; an odd number is guaranteed next.
You press <<


In [ ]:
choice = pipe(prompt)[0]['generated_text'][len(prompt):]
print(choice)

1


# RH 6

In [ ]:
prompt = """You are taking part in a reasoning experiment.
You are participating in a reasoning experiment.
Select what you would do.
Output only <<1>>, <<2>>, or <<3>>.
At a casino, a roulette wheel has landed on red five times in a row.
How would you bet next?
1: Bet on black, expecting a change.
2: Bet on red, expecting the streak to continue.
3: Bet randomly, ignoring prior results.
You press <<"""

print(prompt)

You are taking part in a reasoning experiment.
You are participating in a reasoning experiment.
Select what you would do.
Output only <<1>>, <<2>>, or <<3>>.
At a casino, a roulette wheel has landed on red five times in a row.
How would you bet next?
1: Bet on black, expecting a change.
2: Bet on red, expecting the streak to continue.
3: Bet randomly, ignoring prior results.
You press <<


In [ ]:
choice = pipe(prompt)[0]['generated_text'][len(prompt):]
print(choice)

2


# RH 7-1

In [ ]:
prompt = """You are taking part in a reasoning experiment.
After 10 coin flips (8 heads, 2 tails), Person A concludes the coin is biased toward heads.
Do you agree?
1: Yes, A’s conclusion is correct.
2: No, A’s conclusion is incorrect.
3: Not sure.
You press <<"""

print(prompt)

You are taking part in a reasoning experiment.
After 10 coin flips (8 heads, 2 tails), Person A concludes the coin is biased toward heads.
Do you agree?
1: Yes, A’s conclusion is correct.
2: No, A’s conclusion is incorrect.
3: Not sure.
You press <<


In [ ]:
choice = pipe(prompt)[0]['generated_text'][len(prompt):]
print(choice)

1


# RH 7-2

In [ ]:
prompt = """You are taking part in a reasoning experiment.
The same coin is flipped 1,000 times (520 heads, 480 tails).
Person B concludes the coin is fair.
Do you agree?
1: Yes, B’s conclusion is correct.
2: No, B’s conclusion is incorrect.
3: Not sure.
You press <<"""

print(prompt)

You are taking part in a reasoning experiment.
The same coin is flipped 1,000 times (520 heads, 480 tails).
Person B concludes the coin is fair.
Do you agree?
1: Yes, B’s conclusion is correct.
2: No, B’s conclusion is incorrect.
3: Not sure.
You press <<


In [ ]:
choice = pipe(prompt)[0]['generated_text'][len(prompt):]
print(choice)

1


# RH 8-1-1

In [ ]:
prompt = """You are taking part in a reasoning experiment.
For each teacher description, rate on a scale from 1 (Very Poor) to 10 (Excellent).
Output in the format <<score>>.
Teacher A: The lesson is well-structured and engaging.
How would you rate lesson Quality (1–10)?
You press <<"""

print(prompt)

You are taking part in a reasoning experiment.
For each teacher description, rate on a scale from 1 (Very Poor) to 10 (Excellent).
Output in the format <<score>>.
Teacher A: The lesson is well-structured and engaging.
How would you rate lesson Quality (1–10)? 
You press <<


In [ ]:
choice = pipe(prompt)[0]['generated_text'][len(prompt):]
print(choice)

1


# RH 8-1-2

In [ ]:
prompt = """You are taking part in a reasoning experiment.
For each teacher description, rate on a scale from 1 (Very Poor) to 10 (Excellent).
Output in the format <<score>>.
Teacher A: The lesson is well-structured and engaging.
How would you rate predicted future success in 5 years (1–10)?
You press <<"""

print(prompt)

You are taking part in a reasoning experiment.
For each teacher description, rate on a scale from 1 (Very Poor) to 10 (Excellent).
Output in the format <<score>>.
Teacher A: The lesson is well-structured and engaging.
How would you rate predicted Future Success in 5 years (1–10)?
You press <<


In [ ]:
choice = pipe(prompt)[0]['generated_text'][len(prompt):]
print(choice)

1



``

# RH 8-2-1

In [ ]:
prompt = """You are taking part in a reasoning experiment.
For each teacher description, rate on a scale from 1 (Very Poor) to 10 (Excellent).
Output in the format <<score>>.
Teacher B: The lesson has some good elements but also clear areas for improvement.
How would you rate predicted lesson quality (1–10)?
You press <<"""

print(prompt)

You are taking part in a reasoning experiment.
For each teacher description, rate on a scale from 1 (Very Poor) to 10 (Excellent).
Output in the format <<score>>.
Teacher B: The lesson has some good elements but also clear areas for improvement.
How would you rate predicted lesson quality (1–10)?
You press <<


In [ ]:
choice = pipe(prompt)[0]['generated_text'][len(prompt):]
print(choice)

1


# RH 8-2-2

In [ ]:
prompt = """You are taking part in a reasoning experiment.
For each teacher description, rate on a scale from 1 (Very Poor) to 10 (Excellent).
Output in the format <<score>>.
Teacher B: The lesson has some good elements but also clear areas for improvement.
How would you rate predicted future success in 5 years (1–10)?
You press <<"""

print(prompt)

You are taking part in a reasoning experiment.
For each teacher description, rate on a scale from 1 (Very Poor) to 10 (Excellent).
Output in the format <<score>>.
Teacher B: The lesson has some good elements but also clear areas for improvement.
How would you rate predicted future success in 5 years (1–10)?
You press <<


In [ ]:
choice = pipe(prompt)[0]['generated_text'][len(prompt):]
print(choice)

1


# RH 9

In [ ]:
prompt = """You are taking part in a reasoning experiment.
Read the information and choose the option that matches your intuition.
Output only one choice marker (<<1>> … <<6>>).
Company TechStar has launched a revolutionary product receiving strong positive media coverage.
Its stock price has risen steadily for months.
How likely do you think the stock price will continue increasing next year?
1: Definitely will increase (almost guaranteed)
2: Likely to increase (above 50%)
3: Unsure, could go either way (~50%)
4: Less likely to increase (26–49%)
5: Definitely will decrease (almost guaranteed)
6: Im not sure
You press <<"""

print(prompt)

You are taking part in a reasoning experiment.
Read the information and choose the option that matches your intuition.
Output only one choice marker (<<1>> … <<6>>).
Company TechStar has launched a revolutionary product receiving strong positive media coverage.
Its stock price has risen steadily for months.
How likely do you think the stock price will continue increasing next year?
1: Definitely will increase (almost guaranteed)
2: Likely to increase (above 50%)
3: Unsure, could go either way (~50%)
4: Less likely to increase (26–49%)
5: Definitely will decrease (almost guaranteed)
6: Im not sure
You press <<


In [ ]:
choice = pipe(prompt)[0]['generated_text'][len(prompt):]
print(choice)

1


# RH 10-1

In [ ]:
prompt = """You are taking part in a reasoning experiment.
You are participating in a reasoning experiment.
Rate your confidence level on a scale 1 (Not confident at all) to 5 (Very confident).
Output in the format <<score>>.
You have two students:
- Student A: All B grades in Year 1.
- Student B: A mix of A’s, B’s, and C’s in Year 1.
How confident are you in predicting each student A's final GPA?
You press <<"""

print(prompt)

You are taking part in a reasoning experiment.
You are participating in a reasoning experiment.
Rate your confidence level on a scale 1 (Not confident at all) to 5 (Very confident).
Output in the format <<score>>.
You have two students:
- Student A: All B grades in Year 1.
- Student B: A mix of A’s, B’s, and C’s in Year 1.
How confident are you in predicting each student A's final GPA?
You press <<


In [ ]:
choice = pipe(prompt)[0]['generated_text'][len(prompt):]
print(choice)

1


# RH 10-2

In [ ]:
prompt = """You are taking part in a reasoning experiment.
You are participating in a reasoning experiment.
Rate your confidence level on a scale 1 (Not confident at all) to 5 (Very confident).
Output in the format <<score>>.
You have two students:
- Student A: All B grades in Year 1.
- Student B: A mix of A’s, B’s, and C’s in Year 1.
How confident are you in predicting each student B's final GPA?
You press <<"""

print(prompt)

You are taking part in a reasoning experiment.
You are participating in a reasoning experiment.
Rate your confidence level on a scale 1 (Not confident at all) to 5 (Very confident).
Output in the format <<score>>.
You have two students:
- Student A: All B grades in Year 1.
- Student B: A mix of A’s, B’s, and C’s in Year 1.
How confident are you in predicting each student B's final GPA?
You press <<


In [ ]:
choice = pipe(prompt)[0]['generated_text'][len(prompt):]
print(choice)

1


# RH 11

In [ ]:
prompt = """You are taking part in a reasoning experiment.
Select the statement that best describes your judgment style.
Output only <<1>> or <<2>>.
Do you feel more confident in predictions when input variables are redundant or correlated, even knowing that redundancy may reduce accuracy?
1: Yes, I tend to feel more confident with redundant or correlated inputs.
2: No, I prioritize accuracy and prefer independent inputs.
You press <<"""

print(prompt)

You are taking part in a reasoning experiment.
Select the statement that best describes your judgment style.
Output only <<1>> or <<2>>.
Do you feel more confident in predictions when input variables are redundant or correlated, even knowing that redundancy may reduce accuracy?
1: Yes, I tend to feel more confident with redundant or correlated inputs.
2: No, I prioritize accuracy and prefer independent inputs.
You press <<


In [ ]:
choice = pipe(prompt)[0]['generated_text'][len(prompt):]
print(choice)

1


# RH 12

In [ ]:
prompt = """You are taking part in a reasoning experiment.
Select the statement that seems most plausible to you.
Output only <<1>>, <<2>>, <<3>>, or <<4>>.
A group of students took two similar math tests.
We select the 10 students with the highest scores on the first test.
What will their scores likely be on the second test?
1: Even higher on the second test.
2: About the same.
3: Lower but still above average.
4: Completely unpredictable.

You press <<"""

print(prompt)

You are taking part in a reasoning experiment.
Select the statement that seems most plausible to you.
Output only <<1>>, <<2>>, <<3>>, or <<4>>.
A group of students took two similar math tests.
We select the 10 students with the highest scores on the first test.
What will their scores likely be on the second test?
1: Even higher on the second test.
2: About the same.
3: Lower but still above average.
4: Completely unpredictable.

You press <<


In [ ]:
choice = pipe(prompt)[0]['generated_text'][len(prompt):]
print(choice)

2
